In [11]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset：260次元特徴 + 相対速度 --------
class RelativeSpeedDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []

        print("📥 距離ファイル読み込み中...")
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue

            sid = fname.replace(".json", "")
            print(f"📂 処理中: {sid}")

            if sid not in self.distances:
                print(f"❌ スキップ: 距離情報なし")
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)

            seq = ann.get("sequence", [])
            if len(seq) < 20:
                print(f"⚠️ スキップ: フレーム数 {len(seq)} 未満")
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)

            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)

            if len(dist) < 20:
                print(f"⚠️ スキップ: 距離データが20未満（{len(dist)}）")
                continue

            def smooth(x, w):
                if len(x) < w:
                    return np.zeros_like(x)
                return np.convolve(x, np.ones(w)/w, mode='same')

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]

                if np.any(np.isnan(d)):
                    print(f"❌ NaN in distance @ {sid} frame {i}-{i+19}: {d}")
                    continue
                if np.any(np.isnan(o)):
                    print(f"❌ NaN in own speed @ {sid} frame {i}-{i+19}")
                    continue
                if np.any(np.isnan(t)):
                    print(f"❌ NaN in tgt speed @ {sid} frame {i}-{i+19}")
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.concatenate([
                        d[:20],                      # 20
                        o[:20],                      # 20
                        own_acc[:20],                # 20
                        d1[:20],                     # 20
                        d2[:20],                     # 20
                        f3[:20],                     # 20
                        f5[:20],                     # 20
                        f7[:20],                     # 20
                        f11[:20],                    # 20
                        f11_d1[:20],                 # 20
                        f3[:20] * d1[:20],           # 20
                        f11[:20] - f5[:20],          # 20
                        np.abs(d1[:20]),             # 20
                    ])
                except Exception as e:
                    print(f"❌ 特徴量結合エラー @ {sid} frame {i}: {e}")
                    continue

                if feat.shape[0] != 260:
                    print(f"❌ 特徴量の次元数が不正（{feat.shape[0]}）")
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))
                print(f"✅ 追加: {sid} frame {i} → 合計: {len(self.items)} 件")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid


# -------- シンプルな線形モデル（2層） --------
class SimpleLinear260D(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(260, 64),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)


# -------- 学習ループ --------
def train_simple_model_260d(dataset, save_path="model_260d.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    if len(scenes) == 0:
        raise ValueError("❌ dataset.items が0件です。距離 or アノテーションの不足が原因です。")

    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, sids = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts), list(sids)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleLinear260D().to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    criterion = nn.SmoothL1Loss()

    best_val_loss = float('inf')
    patience = 20
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts, _ in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts, _ in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                loss = criterion(pred, tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step()

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ モデル保存: {save_path}（val_loss={val_loss:.4f}）")
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model


# -------- 実行 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset260D(
        annot_root="./train_annotations",
        distance_json_path="../distance_ref_data.json",
        max_items=7500
    )

    print(f"✅ dataset loaded: {len(dataset)} samples")
    model = train_simple_model_260d(dataset, save_path="model_260d.pth")
    print("✅ 学習完了: model_260d.pth に保存しました")


📥 距離ファイル読み込み中...
📂 処理中: 000
✅ 追加: 000 frame 0 → 合計: 1 件
✅ 追加: 000 frame 1 → 合計: 2 件
✅ 追加: 000 frame 2 → 合計: 3 件
✅ 追加: 000 frame 3 → 合計: 4 件
✅ 追加: 000 frame 4 → 合計: 5 件
✅ 追加: 000 frame 5 → 合計: 6 件
✅ 追加: 000 frame 6 → 合計: 7 件
✅ 追加: 000 frame 7 → 合計: 8 件
✅ 追加: 000 frame 8 → 合計: 9 件
✅ 追加: 000 frame 9 → 合計: 10 件
✅ 追加: 000 frame 10 → 合計: 11 件
✅ 追加: 000 frame 11 → 合計: 12 件
✅ 追加: 000 frame 12 → 合計: 13 件
✅ 追加: 000 frame 13 → 合計: 14 件
✅ 追加: 000 frame 14 → 合計: 15 件
✅ 追加: 000 frame 15 → 合計: 16 件
✅ 追加: 000 frame 16 → 合計: 17 件
✅ 追加: 000 frame 17 → 合計: 18 件
✅ 追加: 000 frame 18 → 合計: 19 件
✅ 追加: 000 frame 19 → 合計: 20 件
✅ 追加: 000 frame 20 → 合計: 21 件
✅ 追加: 000 frame 21 → 合計: 22 件
✅ 追加: 000 frame 22 → 合計: 23 件
✅ 追加: 000 frame 23 → 合計: 24 件
✅ 追加: 000 frame 24 → 合計: 25 件
✅ 追加: 000 frame 25 → 合計: 26 件
✅ 追加: 000 frame 26 → 合計: 27 件
✅ 追加: 000 frame 27 → 合計: 28 件
✅ 追加: 000 frame 28 → 合計: 29 件
✅ 追加: 000 frame 29 → 合計: 30 件
✅ 追加: 000 frame 30 → 合計: 31 件
✅ 追加: 000 frame 31 → 合計: 32 件
✅ 追加: 000 frame 32 → 合計: 33 件
✅

[Train 1]: 100%|██████████| 93/93 [00:00<00:00, 414.63it/s]


Epoch 1 | Train Loss: 1.7805 | Val Loss: 0.6236
✅ モデル保存: model_260d.pth（val_loss=0.6236）


[Train 2]: 100%|██████████| 93/93 [00:00<00:00, 416.84it/s]


Epoch 2 | Train Loss: 0.5635 | Val Loss: 0.3982
✅ モデル保存: model_260d.pth（val_loss=0.3982）


...

Epoch 98 | Train Loss: 0.0941 | Val Loss: 0.0339


[Train 99]: 100%|██████████| 93/93 [00:00<00:00, 428.00it/s]


Epoch 99 | Train Loss: 0.0387 | Val Loss: 0.0160


[Train 100]: 100%|██████████| 93/93 [00:00<00:00, 436.00it/s]

Epoch 100 | Train Loss: 0.0294 | Val Loss: 0.0467
✅ 学習完了: model_260d.pth に保存しました


In [1]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- モデル（学習と同じ構造） --------
class SimpleLinear260D(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(260, 64),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

# -------- 推論用 Dataset（260次元特徴 + 自車速度） --------
class InferenceDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)

            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)):
                    continue

                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)

                try:
                    feat = np.concatenate([
                        d[:20],                      # 20
                        o[:20],                      # 20
                        own_acc[:20],                # 20
                        d1[:20],                     # 20
                        d2[:20],                     # 20
                        f3[:20],                     # 20
                        f5[:20],                     # 20
                        f7[:20],                     # 20
                        f11[:20],                    # 20
                        f11_d1[:20],                 # 20
                        f3[:20] * d1[:20],           # 20
                        f11[:20] - f5[:20],          # 20
                        np.abs(d1[:20]),             # 20
                    ])
                except:
                    continue

                if feat.shape[0] != 260:
                    continue

                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- 推論 + submission.json 作成 --------
def predict_and_save_submission(
    model_path,
    annot_root,
    distance_json_path,
    save_path="submission.json"
):
    dataset = InferenceDataset260D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleLinear260D().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            preds = model(feats).cpu().numpy()
            own_speeds = own_speeds.numpy()
            abs_speeds = preds + own_speeds  # 相対速度 + 自車速度 = 先行車速度

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))  # 20フレーム目に対応

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)

    print(f"✅ 完成: {save_path} に保存しました（scene数: {len(submission)}）")

# -------- 実行部 --------
if __name__ == "__main__":
    predict_and_save_submission(
        model_path="model_260d.pth",
        annot_root="../test2/test_annotations",
        distance_json_path="../testdistance/test_spline_smoothed_fixed.json",
        save_path="submission.json"
    )


100%|██████████| 395/395 [00:00<00:00, 486.49it/s]


✅ 完成: submission.json に保存しました（scene数: 239）
